# Pipeline

https://github.com/datamindedbe/blog-tpcds-dbt-duckdb/tree/main

```
uv sync
```

In [ ]:
# # run this to generate index for values in the hierarchy yaml files

# import duckdb
# from src.hierarchy_duckdb import build_tree_with_stats
# from pathlib import Path
# proj_path = Path().resolve()
# data_path = proj_path / 'data'
# duckdb_conn = duckdb.connect(database=str(proj_path / 'tpcds/tpcds.db'))
# index_path = data_path / 'index'
# for yaml_path in (data_path / 'hierarchy').glob('*.yaml'):
#     tree = build_tree_with_stats(yaml_path, index_path, duckdb_conn)
#     with (data_path / 'hierarchy' / f"{yaml_path.stem}.json").open('w') as f:
#         f.write(tree.to_json())

In [ ]:
# # run this only once to generate the TPC-DS data
import duckdb

con = duckdb.connect(database='./tpcds/tpcds.db')
# con = duckdb.connect(
#     database='./cube-project/data/tpcds.db',
#     read_only=True,
# )
# con.execute('INSTALL tpcds;')
# con.execute('LOAD tpcds;')
# con.execute("CALL dsdgen(sf = 1);")  # run only once generate data with scale factor 1 (1GB)

In [ ]:
df = con.execute("""SELECT ca_zip, Sum(cs_sales_price) 
FROM   catalog_sales, 
       customer, 
       customer_address, 
       date_dim 
WHERE  cs_bill_customer_sk = c_customer_sk 
       AND c_current_addr_sk = ca_address_sk 
       AND ( Substr(ca_zip, 1, 5) IN ( '85669', '86197', '88274', '83405', 
                                       '86475', '85392', '85460', '80348', 
                                       '81792' ) 
              OR ca_state IN ( 'CA', 'WA', 'GA' ) 
              OR cs_sales_price > 500 ) 
       AND cs_sold_date_sk = d_date_sk 
       AND d_qoy = 1 
       AND d_year = 1998 
GROUP  BY ca_zip 
ORDER  BY ca_zip
LIMIT 100; 
""").fetch_df()
df.head()

# Schema Graph

In [ ]:
import sys
from pathlib import Path

proj_path = Path().resolve()
sys.path.append(str(proj_path / 'src'))

from src.graph_vis import display_graph
db_type = 'tutorial'  # 'tutorial' or 'tpcds'
data_path = proj_path / 'data' / db_type


display_graph(data_path, layout='kamada_kawai', height="1000px", width="1200px", 
              legend_toggles_labels=False, 
              node_opacity=1.0,
              node_spacing={'measure': 1.2, 'dimension': 1.1, 'attribute': 0.8, 'default': 0.9},
              node_properties={'fontsize': {'fact': 16, 'dimension': 14, 'default': 14}})

In [4]:
# spring (default): Fruchterman-Reingold force-directed layout.
# kamada_kawai: Kamada-Kawai force-directed layout.
# circular: nodes positioned on a circle.
# shell: concentric shells (fact, dimensions, attributes, measures).
# spectral: based on the graph Laplacian eigenvectors.
# spiral: nodes arranged along an Archimedean spiral.
# random: random placement with a repeatable seed. Legacy Cytoscape names such as radial or cose are automatically mapped to the closest NetworkX algorithm.

display_graph(data_path, layout='kamada_kawai', height="1000px", width="1200px", 
              legend_toggles_labels=False, 
              node_opacity=1.0,
              node_spacing={'measure': 1.2, 'dimension': 1.1, 'attribute': 0.8, 'default': 0.9},
              node_properties={'fontsize': {'fact': 16, 'dimension': 14, 'default': 14}})

2025-11-27 01:56:50.004 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/dim_date.json
2025-11-27 01:56:50.006 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/dim_product.json
2025-11-27 01:56:50.007 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/dim_store.json
2025-11-27 01:56:50.008 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/measure_sales.json


Output()

In [5]:
from src.schema_processor import SchemaExplorer

data_path = proj_path / 'data' / 'tutorial' # 'tutorial' or 'tpcds'
explorer = SchemaExplorer(data_path)
print(explorer.get_facts())
print(explorer.get_schema('star', 'fact_sales'))
print()
# TODO: need from/target searching
attr_results = explorer.search_attribute('star', 'fact_sales', 'year')  # d_fy_year, s_store_id
attr_results

2025-11-27 01:56:55.417 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/dim_date.json
2025-11-27 01:56:55.420 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/dim_product.json
2025-11-27 01:56:55.421 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/dim_store.json
2025-11-27 01:56:55.422 | INFO     | src.schema_processor:_load_hierarchies:140 - Loading hierarchy from /home/jsjang/code/Agent4OLAP/data/tutorial/hierarchy/measure_sales.json


{'fact_sales'}
[{'id': 'fact_sales', 'type': 'fact'}, {'id': 'dim_date', 'type': 'dimension'}, {'id': 'dim_product', 'type': 'dimension'}, {'id': 'dim_store', 'type': 'dimension'}]



[{'dimension': 'dim_date',
  'attribute': 'year',
  'path': [{'type': 'dimension', 'name': 'dim_date', 'label': 'dim_date'},
   {'type': 'level', 'name': 'date', 'label': 'Date'},
   {'type': 'level', 'name': 'month', 'label': 'Month'},
   {'type': 'level', 'name': 'quarter', 'label': 'Quarter'},
   {'type': 'attribute', 'name': 'year', 'label': 'Year'}],
  'stats': {'count': 365,
   'null_count': 0,
   'distinct_count': 1,
   'min': 2024,
   'max': 2024,
   'range': [2024, 2024],
   'dtype': 'integer',
   'unique_values': {'type': 'list', 'values': [2024]}}}]

In [ ]:
import random
from src.unique_index import UniqueIndex
path = './data/tpcds/index/date_dim__d_fy_year'
assert Path(path).exists(), f"Index path {path} does not exist."
idx = UniqueIndex(path, fast=False)

x = random.sample(list(iter(idx)), k=1)[0]
print("Search for:", x)
o = explorer.search_value('star', 'store_sales', 'd_fy_year', x)
print("Found:", o[0]['found'])
o

In [ ]:
o = explorer.search_measure('store_sales', 'sales_price')
o

----

## About the TPC-DS queries

In [ ]:
from pathlib import Path
from collections import defaultdict
from src.schema_processor import SchemaExplorer
queries_path = Path('./queries/tpcds')

fact2queries = defaultdict(list)
queries2fact = defaultdict(set)
for qpath in queries_path.glob('*.sql'):
    with qpath.open() as f:
        sql = f.read()
    for fact in SchemaExplorer.tpcds_facts:
        if fact in sql.lower():
            queries2fact[qpath.stem].add(fact)

for query, facts in queries2fact.items():
    # set the number of the facts as key, more or equal to 4 make them one group
    if len(facts) >= 4:
        fact2queries['4+'].append(query)
    else:
        fact2queries[str(len(facts))].append(query)

In [ ]:
for k, v in sorted(fact2queries.items(), key=lambda x: (int(x[0]) if x[0].isdigit() else 99)):
    print(f"{k}: {len(v)}")

In [ ]:
queries2fact['query15']

In [ ]:
facts_queries_by_numbers: dict[str, dict[str, list[str]]] = defaultdict(dict)
for fact in SchemaExplorer.tpcds_facts:
    for number, queries in fact2queries.items():
        if facts_queries_by_numbers[fact].get(number) is None:
            facts_queries_by_numbers[fact][number] = []
        facts_queries_by_numbers[fact][number].extend(queries)

In [ ]:
sorted(facts_queries_by_numbers['store_sales']['1'])[:5]

In [ ]:
# plot the distribution of number of fact tables per query
# make the number in the center of the bars
# if number is 4 or more, put it in 4+
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('ggplot')

fact_counts = [len(v) for v in queries2fact.values()]
fact_counts = [4 if x >= 4 else x for x in fact_counts]
fig, ax = plt.subplots()

sns.histplot(fact_counts, discrete=True, shrink=0.9, ax=ax)
ax.set_xlabel('Number of Fact Tables in a Query')
ax.set_ylabel('Number of Queries')
ax.set_title('Distribution of Number of Fact Tables per Query')
plt.xticks(ticks=[1, 2, 3, 4], labels=['1', '2', '3', '4+'])
plt.grid(axis='y')
plt.show()

---

# Schema Explorer

In [1]:
import sys
from pathlib import Path

proj_path = Path().resolve()
sys.path.append(str(proj_path / 'src'))

db_type = 'sales'
data_path = proj_path / 'data' / db_type

from src.schema_processor import SchemaExplorer
explorer = SchemaExplorer(data_path, schema_type='star')

2026-01-20 22:41:37.566 | INFO     | src.schema_processor:_load_hierarchies:166 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/sales/hierarchy/dim_date.json
2026-01-20 22:41:37.566 | INFO     | src.schema_processor:_load_hierarchies:166 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/sales/hierarchy/dim_product.json
2026-01-20 22:41:37.567 | INFO     | src.schema_processor:_load_hierarchies:166 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/sales/hierarchy/dim_store.json
2026-01-20 22:41:37.567 | INFO     | src.schema_processor:_load_hierarchies:166 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/sales/hierarchy/measure_sales.json


In [2]:
explorer.get_facts()

['fact_sales']

In [3]:
explorer.get_schema('fact_sales')

[{'name': 'fact_sales',
  'type': 'fact',
  'measures': ['total_units_sold',
   'total_sales_amount',
   'average_unit_price',
   'total_transactions',
   'average_order_value',
   'unique_products_sold',
   'active_stores',
   'active_days',
   'sales_per_store',
   'sales_per_day',
   'revenue_per_unit'],
  'fks': ['fact_sales.date_key = dim_date.date_key',
   'fact_sales.product_key = dim_product.product_key',
   'fact_sales.store_key = dim_store.store_key']},
 {'name': 'dim_date',
  'type': 'dimension',
  'attributes': ['date_key', 'date', 'week', 'month', 'quarter', 'year']},
 {'name': 'dim_product',
  'type': 'dimension',
  'attributes': ['product_key',
   'product_name',
   'brand',
   'type',
   'category',
   'department',
   'marketing_group']},
 {'name': 'dim_store',
  'type': 'dimension',
  'attributes': ['store_key',
   'store_name',
   'sales_manager',
   'sales_district',
   'city',
   'state']}]

In [4]:
explorer.search_attribute('fact_sales', 'state')

[{'dimension': 'dim_store',
  'attribute': 'state',
  'path': [{'type': 'dimension', 'name': 'dim_store', 'label': 'dim_store'},
   {'type': 'level', 'name': 'store_name', 'label': 'Store'},
   {'type': 'level', 'name': 'city', 'label': 'City'},
   {'type': 'attribute', 'name': 'state', 'label': 'State'}],
  'stats': {'count': 20,
   'null_count': 0,
   'distinct_count': 4,
   'min': None,
   'max': None,
   'range': None,
   'dtype': 'string',
   'unique_values': {'type': 'list',
    'values': ['California', 'Florida', 'New York', 'Texas']}}}]

In [5]:
explorer.search_value('fact_sales', 'week', 3)

True

# Agent

https://platform.openai.com/docs/guides/latest-model

https://openai.github.io/openai-agents-python/examples/

In [5]:
from agents import Agent, ModelSettings, function_tool
model_settings = ModelSettings(
    reasoning={"effort": "low"},  # low, medium, high
    verbosity='low',  # low, medium, high
    max_turns=30,
    response_format={
        "type": "json_object",
    },
)

agent_sql = Agent(
    name="OLAP Agent",
    model="gpt-5-nano",
    tools=[], # keep the tool empty for now
    model_settings=model_settings,
)

In [6]:
from agents import function_tool
from src.schema_processor import SchemaExplorer

data_path = Path("./data/sales")
explorer = SchemaExplorer(data_path, schema_type='star')

@function_tool
def get_facts() -> set[str]:
    """returns the list of fact tables."""
    facts = explorer.get_facts()
    return facts

@function_tool
def get_schema_info(fact_table: str) -> list[dict]:
    """returns schema information for the specified fact table."""
    schema = explorer.get_schema(fact_table)
    return schema

@function_tool
def search_attribute(fact_table: str, attribute_name: str) -> list[dict]:
    """searches all hierarchy paths leading to the attribute for the fact schema.
    It returns a list of hierarchy paths and the statistics of the attribute.
    """
    results = explorer.search_attribute(fact_table, attribute_name)
    return results

@function_tool
def search_value_exists(fact_table: str, attribute_name: str, value: str) -> bool:
    """Return whether the attribute value exists in the specified fact table and attribute."""
    exists = explorer.search_value(fact_table, attribute_name, value)
    return exists



2026-01-21 22:15:05.792 | INFO     | src.schema_processor:_load_hierarchies:166 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/sales/hierarchy/dim_date.json
2026-01-21 22:15:05.793 | INFO     | src.schema_processor:_load_hierarchies:166 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/sales/hierarchy/dim_product.json
2026-01-21 22:15:05.794 | INFO     | src.schema_processor:_load_hierarchies:166 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/sales/hierarchy/dim_store.json
2026-01-21 22:15:05.794 | INFO     | src.schema_processor:_load_hierarchies:166 - Loading hierarchy from /Users/soo/code/Agent4OLAP/data/sales/hierarchy/measure_sales.json


In [7]:
import json
from agents import Agent, ModelSettings, function_tool
from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel, Field
_ = load_dotenv(find_dotenv())

from src.schema_processor import SchemaExplorer
from pathlib import Path

class OutputList(BaseModel):
    results: list[str] = Field(..., description="List of results.")
    sql: str | None = Field(None, description="Generated SQL query, if applicable.")

data_path = Path("./data/tutorial")
explorer = SchemaExplorer(data_path, schema_type='star')

model_settings = ModelSettings(
    reasoning={"effort": "low"},  # low, medium, high
    verbosity='low',  # low, medium, high
    max_turns=30,
    response_format={
        "type": "json_object",
        "schema": OutputList.model_json_schema()
    },
)

@function_tool
def get_facts() -> set[str]:
    """returns the list of fact tables."""
    facts = explorer.get_facts()
    return facts

@function_tool
def get_schema_info(fact_table: str) -> list[dict]:
    """returns schema information for the specified fact table."""
    schema = explorer.get_schema(fact_table)
    return schema

@function_tool
def search_attribute(fact_table: str, attribute_name: str) -> list[dict]:
    """searches all hierarchy paths leading to the attribute for the fact schema.
    It returns a list of hierarchy paths and the statistics of the attribute.
    """
    results = explorer.search_attribute(fact_table, attribute_name)
    return results

@function_tool
def search_value_exists(fact_table: str, attribute_name: str, value: str) -> bool:
    """Return whether the attribute value exists in the specified fact table and attribute."""
    exists = explorer.search_value(fact_table, attribute_name, value)
    return exists

instructions_api = """You are an agent that helps users explore OLAP schema information. 
Derive the schema linking from natural language queries to structured fields:
1. measure: a column to be aggregated.
2. dimension: a column used to slice or group the measures.
3. filter: a condition to restrict the data. 

Format: Your output must strictly follow the JSON schema defined in OutputList.
1. Use `<measure:[str, table_name.column_name]>` to denote measures.
2. Use `<dimension:[str, table_name.column_name]>` to denote dimensions.
3. Use `<filter:[str, table_name.column_name]|val:[list[Any], value]>` to denote filters with specific values

For example:
Input: Show me the total sales by each brand for 2025.
Output:
{
    "results": [
        "measure:fact_table.total_sales_amount", 
        "dimension:custom.brand",
        "filter:date.year|val:[2025]"
    ]
}

You have access to the following tools:
1. get_facts(): returns the list of fact tables.
2. get_schema_info(fact_table): returns schema information for the specified fact table.
3. search_attribute(fact_table, attribute_name): searches all hierarchy paths leading to the attribute for the fact schema. It returns a list of hierarchy paths and the statistics of the attribute.
4. search_value_exists(fact_table, attribute_name, value): Return whether the attribute value exists.
Use these tools to answer user queries about the OLAP schema.
"""

instructions_sql = """You are an agent that helps users explore OLAP schema information. 
Derive the schema linking from natural language queries to structured fields first, then convert them to SQL:
1. measure: a column to be aggregated.
2. dimension: a column used to slice or group the measures.
3. filter: a condition to restrict the data. 

Format: Your output must strictly follow the JSON schema defined in OutputList.
1. Use `<measure:[str, table_name.column_name]>` to denote measures.
2. Use `<dimension:[str, table_name.column_name]>` to denote dimensions.
3. Use `<filter:[str, table_name.column_name]|val:[list[Any], value]>` to denote filters with specific values

For example:
Input: Show me the total sales by each brand for 2025.
Output:
{
    "results": [
        "measure:fact_table.total_sales_amount", 
        "dimension:custom.brand",
        "filter:date.year|val:[2025]"
    ]
    "sql": "SELECT custom.brand, SUM(fact_table.total_sales_amount) FROM fact_table JOIN custom ON fact_table.custom_id = custom.custom_key JOIN date ON fact_table.date_id = date.id WHERE date.year = 2025 GROUP BY custom.brand"
}

You have access to the following tools:
1. get_facts(): returns the list of fact tables.
2. get_schema_info(fact_table): returns schema information for the specified fact table.
3. search_attribute(fact_table, attribute_name): searches all hierarchy paths leading to the attribute for the fact schema. It returns a list of hierarchy paths and the statistics of the attribute.
4. search_value_exists(fact_table, attribute_name, value): Return whether the attribute value exists.
Use these tools to answer user queries about the OLAP schema.
"""


agent_api = Agent(
    name="OLAP Agent",
    instructions=instructions_api,
    model="gpt-5-nano",
    tools=[
        get_facts,
        get_schema_info,
        search_attribute,
        search_value_exists,
    ],
    model_settings=model_settings,
)


agent_sql = Agent(
    name="OLAP Agent",
    instructions=instructions_sql,
    model="gpt-5-nano",
    tools=[
        get_facts,
        get_schema_info,
        search_attribute,
        search_value_exists,
    ],
    model_settings=model_settings,
)

In [8]:
from agents import Runner

query = "Show me the total sales for the east sales district by each quarter of 2024."
inputs=[
    {
        "role": "user", 
        "content": query
    }
]
result_api = await Runner.run(agent_api, inputs)
result_sql = await Runner.run(agent_sql, inputs)

In [68]:
from agents import RunItemStreamEvent
stream = Runner.run_streamed(agent_sql, inputs)
events = []
async for event in stream.stream_events():
    if isinstance(event, RunItemStreamEvent):
        events.append(event)
        if event.name == 'reasoning_item_created':
            print(f"[{event.item.type}]: OpenAI hides the content")
        if event.name == 'tool_called':
            print(f"[{event.item.type}]: Calling tool `{event.item.raw_item.name}(**{event.item.raw_item.arguments})`")
        if event.name == 'tool_output':
            print(f"[{event.item.type}]: {event.item.output}")
        if event.name == 'message_output_created':
            print(f"[{event.item.type}]: {event.item.raw_item.content[0].text}")

[reasoning_item]: OpenAI hides the content
[tool_call_item]: Calling tool `get_facts(**{})`
[tool_call_output_item]: ['fact_sales']
[reasoning_item]: OpenAI hides the content
[tool_call_item]: Calling tool `get_schema_info(**{"fact_table":"fact_sales"})`
[tool_call_output_item]: An error occurred while running the tool. Please try again. Error: 'fact_sales not in graph_dict keys: []'
[reasoning_item]: OpenAI hides the content
[tool_call_item]: Calling tool `get_schema_info(**{"fact_table":"fact_sales"})`
[tool_call_output_item]: An error occurred while running the tool. Please try again. Error: 'fact_sales not in graph_dict keys: []'
[reasoning_item]: OpenAI hides the content
[tool_call_item]: Calling tool `get_schema_info(**{"fact_table":"fact_sales"})`
[tool_call_output_item]: An error occurred while running the tool. Please try again. Error: 'fact_sales not in graph_dict keys: []'
[reasoning_item]: OpenAI hides the content
[tool_call_item]: Calling tool `search_attribute(**{"fact_ta

In [67]:
events[1].item.raw_item.arguments

'{}'

In [55]:
event.item.output

['fact_sales']

In [ ]:
for item in result_sql.new_items:
    if item.type == 'reasoning_item':
        print(f"[{item.type}]: OpenAI hides the content")
    if item.type == 'tool_call_item':
        print(f"[{item.type}]: {item.raw_item.name}")
    if item.type == 'tool_call_output_item':
        print(f"[{item.type}]: {item.raw_item['output']}")

In [21]:
event

RunItemStreamEvent(name='message_output_created', item=MessageOutputItem(agent=Agent(name='OLAP Agent', handoff_description=None, tools=[FunctionTool(name='get_facts', description='returns the list of fact tables.', params_json_schema={'properties': {}, 'title': 'get_facts_args', 'type': 'object', 'additionalProperties': False, 'required': []}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11d48a840>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='get_schema_info', description='returns schema information for the specified fact table.', params_json_schema={'properties': {'fact_table': {'title': 'Fact Table', 'type': 'string'}}, 'required': ['fact_table'], 'title': 'get_schema_info_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x106d13060>, strict_json_sche

In [15]:
x = result_api.final_output
print(json.loads(x)['results'])
print("-----"*10)

x = result_sql.final_output.replace('\\\n', '')
print(json.loads(x)['results'])
print(json.loads(x)['sql'])

['measure:fact_sales.total_sales_amount', 'dimension:dim_date.quarter', 'dimension:dim_store.sales_district', 'filter:dim_date.year|val:[2024]', 'filter:dim_store.sales_district|val:[East]']
--------------------------------------------------
['measure:fact_sales.total_sales_amount', 'dimension:dim_date.quarter', 'filter:dim_date.year|val:[2024]', 'filter:dim_store.sales_district|val:[East]']
SELECT dim_date.quarter, SUM(fact_sales.total_sales_amount) AS total_sales_amount FROM fact_sales JOIN dim_date ON fact_sales.date_key = dim_date.date_key JOIN dim_store ON fact_sales.store_key = dim_store.store_key WHERE dim_date.year = 2024 AND dim_store.sales_district = 'East' GROUP BY dim_date.quarter ORDER BY dim_date.quarter


In [16]:
for item in result_api.new_items:
    if item.type == 'reasoning_item':
        print(f"[{item.type}]: OpenAI hides the content")
    if item.type == 'tool_call_item':
        print(f"[{item.type}]: {item.raw_item.name}")
    if item.type == 'tool_call_output_item':
        print(f"[{item.type}]: {item.raw_item['output']}")

[reasoning_item]: OpenAI hides the content
[tool_call_item]: get_facts
[tool_call_output_item]: ['fact_sales']
[reasoning_item]: OpenAI hides the content
[tool_call_item]: get_schema_info
[tool_call_output_item]: [{'name': 'fact_sales', 'type': 'fact', 'measures': ['total_units_sold', 'total_sales_amount', 'average_unit_price', 'total_transactions', 'average_order_value', 'unique_products_sold', 'active_stores', 'active_days', 'sales_per_store', 'sales_per_day', 'revenue_per_unit'], 'fks': ['fact_sales.date_key = dim_date.date_key', 'fact_sales.product_key = dim_product.product_key', 'fact_sales.store_key = dim_store.store_key']}, {'name': 'dim_date', 'type': 'dimension', 'attributes': ['date_key', 'date', 'week', 'month', 'quarter', 'year']}, {'name': 'dim_product', 'type': 'dimension', 'attributes': ['product_key', 'product_name', 'brand', 'type', 'category', 'department', 'marketing_group']}, {'name': 'dim_store', 'type': 'dimension', 'attributes': ['store_key', 'store_name', 'sales

In [17]:
for item in result_sql.new_items:
    if item.type == 'reasoning_item':
        print(f"[{item.type}]: OpenAI hides the content")
    if item.type == 'tool_call_item':
        print(f"[{item.type}]: {item.raw_item.name}")
    if item.type == 'tool_call_output_item':
        print(f"[{item.type}]: {item.raw_item['output']}")

[reasoning_item]: OpenAI hides the content
[tool_call_item]: get_facts
[tool_call_output_item]: ['fact_sales']
[reasoning_item]: OpenAI hides the content
[tool_call_item]: get_schema_info
[tool_call_output_item]: [{'name': 'fact_sales', 'type': 'fact', 'measures': ['total_units_sold', 'total_sales_amount', 'average_unit_price', 'total_transactions', 'average_order_value', 'unique_products_sold', 'active_stores', 'active_days', 'sales_per_store', 'sales_per_day', 'revenue_per_unit'], 'fks': ['fact_sales.date_key = dim_date.date_key', 'fact_sales.product_key = dim_product.product_key', 'fact_sales.store_key = dim_store.store_key']}, {'name': 'dim_date', 'type': 'dimension', 'attributes': ['date_key', 'date', 'week', 'month', 'quarter', 'year']}, {'name': 'dim_product', 'type': 'dimension', 'attributes': ['product_key', 'product_name', 'brand', 'type', 'category', 'department', 'marketing_group']}, {'name': 'dim_store', 'type': 'dimension', 'attributes': ['store_key', 'store_name', 'sales

# Event Logs

In [5]:
# import duckdb
# from pathlib import Path

# con = duckdb.connect(str(Path("data/logs/events.duckdb")))
# con.execute("DELETE FROM superset_action_logs")
# con.execute("DELETE FROM _checkpoint WHERE key='superset_last_id'")
# con.close()

In [6]:
import duckdb
from pathlib import Path
from IPython.display import display
import duckdb

def refresh_conn(path="data/logs/events.duckdb"):
    try:
        con.close()
    except Exception:
        pass
    return duckdb.connect(path, read_only=True)

con = refresh_conn()
df = con.execute("SELECT * FROM superset_action_logs ORDER BY dttm DESC LIMIT 50").df()
df.head()

,superset_log_id,dttm,action,user_id,dashboard_id,slice_id,duration_ms,referrer,json,ingested_at
0,15864,2026-01-17 18:17:43.752448,ChartDataRestApi.data,<NA>,12,314,140,http://localhost:8088/embedded/2c2df33c-bbe2-4...,"{""path"": ""/api/v1/chart/data"", ""form_data"": {""...",2026-01-17 18:17:44.308538
1,15863,2026-01-17 18:17:43.751583,_get_data_response,<NA>,12,314,107,http://localhost:8088/embedded/2c2df33c-bbe2-4...,None,2026-01-17 18:17:44.308536
2,15862,2026-01-17 18:17:43.750238,ChartDataRestApi.json_dumps,<NA>,12,314,0,http://localhost:8088/embedded/2c2df33c-bbe2-4...,"{""path"": ""/api/v1/chart/data"", ""form_data"": {""...",2026-01-17 18:17:44.308534
3,15861,2026-01-17 18:17:43.743337,QueryObject.post_processing,<NA>,12,314,0,http://localhost:8088/embedded/2c2df33c-bbe2-4...,"{""path"": ""/api/v1/chart/data"", ""form_data"": {""...",2026-01-17 18:17:44.308532
4,15860,2026-01-17 18:17:43.738618,load_into_dataframe,<NA>,12,314,1,http://localhost:8088/embedded/2c2df33c-bbe2-4...,"{""path"": ""/api/v1/chart/data"", ""form_data"": {""...",2026-01-17 18:17:44.308529


In [7]:
sql = """SELECT * FROM superset_action_logs;"""
df = con.execute(sql).fetchdf()
print(df.shape)
display(df['action'].value_counts())
# print(con.execute("SELECT * FROM superset_action_logs ORDER BY ingested_at DESC LIMIT 5").fetchall())

(10057, 10)


action
execute_sql                    1440
fetch_rows                     1440
load_into_dataframe            1440
QueryObject.post_processing    1440
ChartDataRestApi.data          1156
ChartDataRestApi.json_dumps    1154
_get_data_response             1154
DashboardRestApi.get            646
log                             174
dashboard                         8
ExploreRestApi.get                5
Name: count, dtype: int64

In [8]:
df.to_csv('superset_action_logs.csv', index=False)

In [4]:
for idx, row in df.iterrows():
    if row['json'] is not None:
        log_json = json.loads(row['json'])
        if log_json.get('queries', {}):
            print(f'== [{idx}]', row['action'], log_json.keys())
            print(log_json.get('queries', {}))
            for x in log_json.get('queries', {}):
                print(x.keys())

NameError: name 'json' is not defined

In [45]:
import json
chart_id = 314
row = con.execute(
    """
    SELECT superset_log_id, dttm, action, dashboard_id, slice_id, json
    FROM superset_action_logs
    WHERE slice_id = ? AND action IN ('ChartDataRestApi.data', 'ChartDataRestApi.json_dumps')
    ORDER BY superset_log_id DESC
    LIMIT 1
    """,
    [chart_id],
).fetchone()
payload_json = row[5] or ""
try:
    payload = json.loads(payload_json) if payload_json else {}
except json.JSONDecodeError:
    payload = {}

In [46]:
payload['datasource']

{'id': 27, 'type': 'table'}

In [ ]:
# j = log_json
j = payload
chart_payload = {
  "datasource": j["datasource"],
  "queries": j["queries"],
  "result_format": j["result_format"],
  "result_type": j["result_type"],
  "force": j.get("force", False),

  "form_data": j.get("form_data")
}

In [49]:
import json
import requests

def superset_login(host: str, username: str, password: str) -> tuple[requests.Session, str]:
    """
    Returns (session, access_token)
    - session: 쿠키 유지용 (CSRF/세션 요구 환경에서 필요)
    - access_token: Authorization: Bearer 용
    """
    s = requests.Session()
    url = f"{host}/api/v1/security/login"
    payload = {
        "username": username,
        "password": password,
        "provider": "db",
        "refresh": True,
    }
    r = s.post(url, json=payload, timeout=30)
    r.raise_for_status()
    data = r.json()
    access_token = data["access_token"]
    return s, access_token

def get_csrf_token(host: str, session: requests.Session, access_token: str) -> str:
    url = f"{host}/api/v1/security/csrf_token/"
    headers = {"Authorization": f"Bearer {access_token}"}
    r = session.get(url, headers=headers, timeout=30)
    r.raise_for_status()
    return r.json()["result"]

def get_csrf_token(host: str, session: requests.Session, access_token: str) -> str:
    """
    일부 환경에서는 POST 요청에 CSRF 토큰을 요구함.
    Session 쿠키도 함께 유지되어야 하는 경우가 많아 session을 사용.
    """
    url = f"{host}/api/v1/security/csrf_token/"
    headers = {"Authorization": f"Bearer {access_token}"}
    r = session.get(url, headers=headers, timeout=30)
    r.raise_for_status()
    return r.json()["result"]


def fetch_chart_data(
    host: str,
    session: requests.Session,
    access_token: str,
    chart_payload: dict,
    csrf_token: str | None = None,
) -> dict:
    """
    chart_payload는 로그의 ChartDataRestApi.data json에서
    datasource/queries/result_format/result_type/force/(optional)form_data 형태로 만든 payload.
    """
    url = f"{host}/api/v1/chart/data"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }
    if csrf_token:
        headers["X-CSRFToken"] = csrf_token

    r = session.post(url, headers=headers, json=chart_payload, timeout=120)
    r.raise_for_status()
    return r.json()


if __name__ == "__main__":
    HOST = "http://localhost:8088"  # 예: https://superset.mycompany.com
    USERNAME = "harry_potter"
    PASSWORD = "1234"
    session, access = superset_login(HOST, USERNAME, PASSWORD)
    csrf = get_csrf_token(HOST, session, access)

    # chart_payload 는 json으로부터 가져옴
    # j = log_json
    # chart_payload = {
    # "datasource": j["datasource"],
    # "queries": j["queries"],
    # "result_format": j["result_format"],
    # "result_type": j["result_type"],
    # "force": j.get("force", False),

    # "form_data": j.get("form_data")
    # }
    resp = fetch_chart_data(HOST, session, access, chart_payload, csrf_token=csrf)

    # 5)
    print(resp['result'][0]['data'])


[{'dim_product_marketing_group': 'A', 'total_receipts': 5915984.77, 'store_key': 20, '%total_receipts': 0.6103445651854661, '%store_key': 1.0}, {'dim_product_marketing_group': 'B', 'total_receipts': 1931173.1, 'store_key': 20, '%total_receipts': 0.19923665321020914, '%store_key': 1.0}, {'dim_product_marketing_group': 'C', 'total_receipts': 1845702.7, 'store_key': 20, '%total_receipts': 0.19041878160432466, '%store_key': 1.0}]


In [54]:
import pandas as pd
for x in resp['result']:
    display(pd.DataFrame(x['data']))

,dim_product_marketing_group,total_receipts,store_key,%total_receipts,%store_key
0,A,5915984.77,20,0.610345,1.0
1,B,1931173.10,20,0.199237,1.0
2,C,1845702.70,20,0.190419,1.0


,total_receipts,store_key
0,9692860.57,20


In [33]:
session = requests.Session()
session, access = superset_login(HOST, USERNAME, PASSWORD)
csrf = get_csrf_token(HOST, session, access)

headers = {
    "Authorization": f"Bearer {access}",
    "Content-Type": "application/json",
    "X-CSRFToken": csrf,
}
response = session.get(
    f"{HOST}/api/v1/chart/{chart_id}",
    timeout=30,
    headers=headers,
)
response.raise_for_status()
payload = response.json()
payload["result"]

{'cache_timeout': None,
 'certification_details': None,
 'certified_by': None,
 'changed_on_delta_humanized': '2 days ago',
 'dashboards': [{'dashboard_title': 'Company',
   'id': 12,
   'json_metadata': '{"label_colors": {}, "chart_configuration": {"160": {"id": 160, "crossFilters": {"scope": "global", "chartsInScope": [313, 314, 315, 316]}}, "313": {"id": 313, "crossFilters": {"scope": "global", "chartsInScope": [160, 314, 315, 316]}}, "314": {"id": 314, "crossFilters": {"scope": "global", "chartsInScope": [160, 313, 315, 316]}}, "315": {"id": 315, "crossFilters": {"scope": "global", "chartsInScope": [160, 313, 314, 316]}}, "316": {"id": 316, "crossFilters": {"scope": "global", "chartsInScope": [160, 313, 314, 315]}}}, "global_chart_configuration": {"scope": {"rootPath": ["ROOT_ID"], "excluded": []}, "chartsInScope": [160, 313, 314, 315, 316]}, "color_scheme": "", "refresh_frequency": 0, "expanded_slices": {}, "timed_refresh_immune_slices": [], "cross_filters_enabled": true, "default

In [7]:
j = log_json
payload = {
  "datasource": j["datasource"],
  "queries": j["queries"],
  "result_format": j["result_format"],
  "result_type": j["result_type"],
  "force": j.get("force", False),

  "form_data": j.get("form_data")
}

{'datasource': {'id': 27, 'type': 'table'},
 'queries': [{'filters': [],
   'extras': {'having': '', 'where': ''},
   'applied_time_extras': {},
   'columns': [{'expressionType': 'SQL',
     'label': 'dim_product_marketing_group',
     'sqlExpression': 'dim_product_marketing_group'}],
   'metrics': ['total_receipts', 'store_key'],
   'orderby': [['total_receipts', False]],
   'annotation_layers': [],
   'row_limit': 10000,
   'series_limit': 0,
   'group_others_when_limit_reached': False,
   'order_desc': True,
   'url_params': {'uiConfig': '0', 'expand_filters': 'true'},
   'custom_params': {},
   'custom_form_data': {},
   'post_processing': [{'operation': 'contribution',
     'options': {'columns': ['total_receipts', 'store_key'],
      'rename_columns': ['%total_receipts', '%store_key']}}],
   'time_offsets': []},
  {'filters': [],
   'extras': {'having': '', 'where': ''},
   'applied_time_extras': {},
   'columns': [],
   'metrics': ['total_receipts', 'store_key'],
   'annotation_

In [6]:
cd = df[df["action"] == "ChartDataRestApi.data"].copy()

def parse(s):
    try:
        return json.loads(s)
    except Exception:
        return {}

def extract_filters(j):
    # 우선 queries[].filters
    qlist = j.get("queries") or []
    if not qlist and isinstance(j.get("form_data"), dict):
        qlist = j["form_data"].get("queries") or []
    if not qlist and isinstance(j.get("form_data"), dict):
        # fallback: form_data.filters
        return j["form_data"].get("filters") or []
    if qlist:
        return (qlist[0].get("filters") or [])
    return []

def is_meaningful(filters):
    if not filters:
        return False
    # TEMPORAL_RANGE "No filter" 같은 건 제외하고, IN/== 등의 실제 필터만 남김
    for f in filters:
        if isinstance(f, dict):
            op = f.get("op")
            val = f.get("val")
            if op and not (op == "TEMPORAL_RANGE" and str(val).strip().lower() in ["no filter", "none", ""]):
                return True
    return False

cd["j"] = cd["json"].apply(parse)
cd["filters"] = cd["j"].apply(extract_filters)
filtered = cd[cd["filters"].apply(is_meaningful)]

# 핵심 결과
print("meaningful filter calls:", len(filtered))
print(filtered[["dttm","user_id","dashboard_id","slice_id","filters"]].head(20).to_string(index=False))


meaningful filter calls: 126
                      dttm  user_id  dashboard_id  slice_id                                                                                                                                                                                               filters
2026-01-14 13:05:20.182956     <NA>            12       313                                                               [{'col': 'dim_product_department', 'op': 'IN', 'val': ['Sales']}, {'col': 'dim_date_date', 'op': 'TEMPORAL_RANGE', 'val': 'No filter'}]
2026-01-14 13:05:25.781547     <NA>            12       313                                                           [{'col': 'dim_product_department', 'op': 'IN', 'val': ['Marketing']}, {'col': 'dim_date_date', 'op': 'TEMPORAL_RANGE', 'val': 'No filter'}]
2026-01-14 13:05:27.672159     <NA>            12       313                                                                [{'col': 'dim_product_department', 'op': 'IN', 'val': ['Tech']}, {'col': '

In [9]:
cd['json'][4]

'{"path": "/api/v1/chart/data", "form_data": {"filters": [{"col": "dim_date_date", "op": "TEMPORAL_RANGE", "val": "No filter"}], "extras": {"having": "", "where": ""}, "applied_time_extras": {}, "columns": ["dim_product_department"], "metrics": [{"aggregate": "SUM", "column": {"advanced_data_type": null, "certification_details": null, "certified_by": null, "column_name": "total_receipts", "description": null, "expression": null, "filterable": true, "groupby": true, "id": 771, "is_certified": false, "is_dttm": false, "python_date_format": null, "type": "column", "type_generic": 0, "uuid": "25690c12-f360-4a20-95ce-094d75855d8d", "verbose_name": null, "warning_markdown": null}, "datasourceWarning": false, "expressionType": "SIMPLE", "hasCustomLabel": false, "label": "SUM(total_receipts)", "optionName": "metric_y7b9yx7di6_h9s06w7m27", "sqlExpression": null}], "orderby": [[{"aggregate": "SUM", "column": {"advanced_data_type": null, "certification_details": null, "certified_by": null, "colum

In [ ]:
[{'filters': [{'col': 'dim_date_date', 'op': 'TEMPORAL_RANGE', 'val': 'No filter'}], 'extras': {'having': '', 'where': ''}, 'applied_time_extras': {}, 'columns': ['dim_product_department'], 'metrics': [{'aggregate': 'SUM', 'column': {'advanced_data_type': None, 'certification_details': None, 'certified_by': None, 'column_name': 'total_receipts', 'description': None, 'expression': None, 'filterable': True, 'groupby': True, 'id': 771, 'is_certified': False, 'is_dttm': False, 'python_date_format': None, 'type': 'column', 'type_generic': 0, 'uuid': '25690c12-f360-4a20-95ce-094d75855d8d', 'verbose_name': None, 'warning_markdown': None}, 'datasourceWarning': False, 'expressionType': 'SIMPLE', 'hasCustomLabel': False, 'label': 'SUM(total_receipts)', 'optionName': 'metric_y7b9yx7di6_h9s06w7m27', 'sqlExpression': None}], 'orderby': [[{'aggregate': 'SUM', 'column': {'advanced_data_type': None, 'certification_details': None, 'certified_by': None, 'column_name': 'total_receipts', 'description': None, 'expression': None, 'filterable': True, 'groupby': True, 'id': 771, 'is_certified': False, 'is_dttm': False, 'python_date_format': None, 'type': 'column', 'type_generic': 0, 'uuid': '25690c12-f360-4a20-95ce-094d75855d8d', 'verbose_name': None, 'warning_markdown': None}, 'datasourceWarning': False, 'expressionType': 'SIMPLE', 'hasCustomLabel': False, 'label': 'SUM(total_receipts)', 'optionName': 'metric_y7b9yx7di6_h9s06w7m27', 'sqlExpression': None}, False]], 'annotation_layers': [], 'row_limit': 5000, 'series_limit': 0, 'group_others_when_limit_reached': False, 'order_desc': True, 'url_params': {}, 'custom_params': {}, 'custom_form_data': {}, 'post_processing': [{'operation': 'contribution', 'options': {'columns': ['SUM(total_receipts)'], 'rename_columns': ['SUM(total_receipts)__contribution']}}]}]


In [ ]:
import duckdb
from pathlib import Path
from IPython.display import display
db_path = Path('data/logs/events.duckdb')
con = duckdb.connect(str(db_path))
sql = """SELECT * FROM superset_action_logs;"""
df = con.execute(sql).fetchdf()
print(df.shape)
display(df['action'].value_counts())
# print(con.execute("SELECT * FROM superset_action_logs ORDER BY ingested_at DESC LIMIT 5").fetchall())

(532, 10)


action
DashboardRestApi.get           120
log                             67
ChartDataRestApi.json_dumps     61
_get_data_response              61
ChartDataRestApi.data           61
execute_sql                     39
fetch_rows                      39
load_into_dataframe             39
QueryObject.post_processing     39
dashboard                        5
ExploreRestApi.get               1
Name: count, dtype: int64

In [18]:
from typing import Any, Dict, List, Optional, Set, Tuple

# ---------- mapping helpers ----------
def _map_column(expr: str) -> Tuple[str, Optional[str]]:
    """
    Superset form_data의 컬럼명(SQL expression) -> (sql_expr, join_needed)
    join_needed: 'dim_date' | 'dim_product' | 'dim_store' | None
    """
    if expr.startswith("dim_product_"):
        col = expr.replace("dim_product_", "", 1)
        return f"dp.{col}", "dim_product"
    if expr.startswith("dim_date_"):
        col = expr.replace("dim_date_", "", 1)
        return f"dd.{col}", "dim_date"
    if expr.startswith("dim_store_"):
        col = expr.replace("dim_store_", "", 1)
        return f"ds.{col}", "dim_store"
    # default: fact
    return f"fs.{expr}", None


def _metric_sql(m: Any) -> Tuple[str, str, Set[str]]:
    """
    metric definition -> (select_expr, alias, joins_needed)
    - metric이 dict면: {aggregate: 'SUM', column: {column_name: 'total_receipts'}, label: 'SUM(Revenue)'} 형식 지원
    - metric이 string이면 heuristic:
        - *_key => COUNT(DISTINCT ...)
        - 그 외 => SUM(...)
    """
    joins: Set[str] = set()

    # dict metric (Superset UI metric object)
    if isinstance(m, dict):
        agg = (m.get("aggregate") or "").upper()
        label = m.get("label") or "metric"
        col_obj = m.get("column") or {}
        col_name = col_obj.get("column_name") or col_obj.get("label") or col_obj.get("sqlExpression") or ""

        col_sql, j = _map_column(col_name)
        if j:
            joins.add(j)

        if not agg:
            # fallback: treat as raw expression
            return f"{col_sql} AS \"{label}\"", label, joins
        return f"{agg}({col_sql}) AS \"{label}\"", label, joins

    # string metric
    if isinstance(m, str):
        name = m
        col_sql, j = _map_column(name)
        if j:
            joins.add(j)

        # heuristic: *_key => count distinct
        if name.endswith("_key"):
            alias = name
            return f"COUNT(DISTINCT {col_sql}) AS \"{alias}\"", alias, joins

        alias = name
        return f"SUM({col_sql}) AS \"{alias}\"", alias, joins

    # unknown
    return "NULL AS \"metric\"", "metric", joins


def _orderby_sql(orderby: Any, metric_aliases: Set[str], force_desc: bool) -> str:
    """
    orderby 예:
      - [['total_receipts', False]]
      - [[{metric_obj...}, False]]
    """
    if not orderby:
        return ""

    item = orderby[0]
    if not isinstance(item, (list, tuple)) or len(item) < 2:
        return ""

    key, asc_flag = item[0], item[1]
    direction = "DESC" if force_desc else ("ASC" if asc_flag else "DESC")

    # key가 dict(metric obj)인 경우 label을 ORDER BY 대상으로
    if isinstance(key, dict):
        label = key.get("label") or "metric"
        return f"ORDER BY \"{label}\" {direction}"

    # key가 string이면 metric alias 우선, 아니면 컬럼 매핑
    if isinstance(key, str):
        if key in metric_aliases:
            return f"ORDER BY \"{key}\" {direction}"
        col_sql, _ = _map_column(key)
        return f"ORDER BY {col_sql} {direction}"

    return ""


def _build_sql_for_query(q: Dict[str, Any], default_schema: str = "main") -> str:
    joins_needed: Set[str] = set()

    # columns (groupby)
    cols = []
    groupbys = []

    for c in q.get("columns", []) or []:
        # logs에서 column object: {'sqlExpression': 'dim_product_marketing_group', 'label': ...}
        if isinstance(c, dict):
            expr = c.get("sqlExpression") or c.get("label") or ""
            label = c.get("label") or expr
        else:
            expr = str(c)
            label = expr

        if not expr:
            continue

        col_sql, j = _map_column(expr)
        if j:
            joins_needed.add(j)

        cols.append(f"{col_sql} AS {label}")
        groupbys.append(col_sql)

    # metrics
    metric_aliases: Set[str] = set()
    metric_selects = []
    for m in (q.get("metrics") or []):
        sel, alias, jset = _metric_sql(m)
        metric_selects.append(sel)
        metric_aliases.add(alias)
        joins_needed |= jset

    select_parts = cols + metric_selects
    if not select_parts:
        select_parts = ["*"]

    # FROM + joins
    from_sql = f"FROM {default_schema}.fact_sales fs"
    join_sql_parts = []
    if "dim_date" in joins_needed:
        join_sql_parts.append(f"LEFT JOIN {default_schema}.dim_date dd ON fs.date_key = dd.date_key")
    if "dim_product" in joins_needed:
        join_sql_parts.append(f"LEFT JOIN {default_schema}.dim_product dp ON fs.product_key = dp.product_key")
    if "dim_store" in joins_needed:
        join_sql_parts.append(f"LEFT JOIN {default_schema}.dim_store ds ON fs.store_key = ds.store_key")

    # WHERE/HAVING (extras)
    extras = q.get("extras") or {}
    where_txt = (extras.get("where") or "").strip()
    having_txt = (extras.get("having") or "").strip()

    where_sql = f"WHERE {where_txt}" if where_txt else ""
    groupby_sql = f"GROUP BY {', '.join(groupbys)}" if groupbys else ""
    having_sql = f"HAVING {having_txt}" if (having_txt and groupbys) else ""

    # ORDER BY
    force_desc = bool(q.get("order_desc"))
    orderby_sql = _orderby_sql(q.get("orderby"), metric_aliases, force_desc)

    # LIMIT (row_limit=0이면 보통 limit 없음으로 처리)
    row_limit = q.get("row_limit")
    limit_sql = f"LIMIT {int(row_limit)}" if isinstance(row_limit, int) and row_limit > 0 else ""

    # stitch
    sql = "SELECT\n  " + ",\n  ".join(select_parts) + "\n" + from_sql
    if join_sql_parts:
        sql += "\n" + "\n".join(join_sql_parts)
    if where_sql:
        sql += "\n" + where_sql
    if groupby_sql:
        sql += "\n" + groupby_sql
    if having_sql:
        sql += "\n" + having_sql
    if orderby_sql:
        sql += "\n" + orderby_sql
    if limit_sql:
        sql += "\n" + limit_sql
    sql += ";"
    return sql


def fetch_sql(log_queries: List[Dict[str, Any]], default_schema: str = "main") -> List[str]:
    """
    Superset action log에서 뽑은 queries(list[dict])를 SQL list로 변환
    """
    return [_build_sql_for_query(q, default_schema=default_schema) for q in log_queries]

In [24]:
for idx, row in df_temp.iterrows():
    
    if row['action'] == 'execute_sql':
        print("-----"*10 + f" Row idx: {idx} " + "-----"*10)
        if row['json'] is not None:
            log_json = json.loads(row['json'])
            sqls = fetch_sql(log_json.get('queries', []), default_schema='main')
            for s in sqls:
                print(s)

            

-------------------------------------------------- Row idx: 4235 --------------------------------------------------
SELECT
  dd.year AS dim_date_year,
  dd.quarter AS dim_date_quarter,
  SUM(fs.total_receipts) AS "SUM(Revenue)"
FROM main.fact_sales fs
LEFT JOIN main.dim_date dd ON fs.date_key = dd.date_key
GROUP BY dd.year, dd.quarter
ORDER BY "SUM(Revenue)" DESC
LIMIT 10000;
-------------------------------------------------- Row idx: 4236 --------------------------------------------------
SELECT
  dp.brand AS dim_product_brand,
  SUM(fs.total_receipts) AS "SUM(Revenue)"
FROM main.fact_sales fs
LEFT JOIN main.dim_product dp ON fs.product_key = dp.product_key
GROUP BY dp.brand
ORDER BY "SUM(Revenue)" DESC
LIMIT 10000;
-------------------------------------------------- Row idx: 4241 --------------------------------------------------
SELECT
  dp.marketing_group AS dim_product_marketing_group,
  SUM(fs.total_receipts) AS "total_receipts",
  COUNT(DISTINCT fs.store_key) AS "store_key"
FROM 

In [11]:
df.to_csv('superset_action_logs.csv', index=False)

In [1]:
from __future__ import annotations
import duckdb
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Any

from pydantic import BaseModel, Field

import sys
from pathlib import Path
sys.path.append(str(Path().resolve() / 'fastapi'))

from fastapi_service.superset import (
    fetch_chart_data_from_log,
    fetch_dashboard_charts,
)


In [2]:
try:
    from agents import Agent, ModelSettings, Runner, function_tool
except Exception as exc:  # pragma: no cover - optional dependency
    Agent = None
    ModelSettings = None
    Runner = None
    function_tool = None
    _IMPORT_ERROR = exc
else:
    _IMPORT_ERROR = None

class Answer(BaseModel):
    answer: dict[str, Any] = Field(..., description="Final answer.")


@dataclass
class AgentContext:
    settings: Any
    conn: Any
    chart_id: int | None = None
    chart_name: str | None = None
    chart_data: dict[str, Any] | None = None


_ACTIVE_CONTEXT: AgentContext | None = None


def _get_active_context() -> AgentContext | None:
    return _ACTIVE_CONTEXT


def _fetch_latest_chart_log_payload(
    conn: duckdb.DuckDBPyConnection,
    chart_id: int,
) -> dict[str, Any] | None:
    row = conn.execute(
        """
        SELECT json
        FROM superset_action_logs
        WHERE slice_id = ? AND action IN ('ChartDataRestApi.data', 'ChartDataRestApi.json_dumps')
        ORDER BY superset_log_id DESC
        LIMIT 1
        """,
        [chart_id],
    ).fetchone()
    if not row or not row[0]:
        return None
    try:
        return json.loads(row[0])
    except json.JSONDecodeError:
        return None


if function_tool:

    @function_tool
    def get_active_chart_log() -> dict[str, Any]:
        """
        Return the latest Superset log payload for the active chart.

        Input:
        - Uses the active context set by the API handler (chart id + DuckDB connection).

        Output:
        - {"chart_id": int, "payload": dict | null}
        - {"error": "..."} on missing chart id or missing payload.
        """
        context = _get_active_context()
        if context is None:
            return {"error": "active context not set"}
        if not context.chart_id:
            return {"error": "active chart id not set"}
        payload = _fetch_latest_chart_log_payload(context.conn, context.chart_id)
        return {"chart_id": context.chart_id, "payload": payload}

    @function_tool
    def get_active_chart_data() -> dict[str, Any]:
        """
        Fetch chart data from Superset using the latest log payload.

        Input:
        - Uses the active context set by the API handler (chart id + settings).

        Output:
        - {"chart_id": int, "data": dict} (raw Superset /api/v1/chart/data response)
        - {"error": "..."} on missing chart id or missing payload.
        """
        context = _get_active_context()
        if context is None:
            return {"error": "active context not set"}
        if not context.chart_id:
            return {"error": "active chart id not set"}
        payload = _fetch_latest_chart_log_payload(context.conn, context.chart_id)
        if not payload:
            return {"error": "no chart log payload found"}
        data = fetch_chart_data_from_log(context.settings, payload)
        return {"chart_id": context.chart_id, "data": data}

    @function_tool
    def get_chart_sql() -> dict[str, Any]:
        """
        Return the latest SQL/query for the active chart from DuckDB logs.

        Input:
        - Uses the active context set by the API handler (chart id + DuckDB connection).

        Output:
        - {"chart_id": int, "sql": str | null}
        - {"error": "..."} if no log payload or invalid JSON.
        """
        context = _get_active_context()
        if context is None:
            return {"error": "active context not set"}
        if not context.chart_id:
            return {"error": "active chart id not set"}
        row = context.conn.execute(
            """
            SELECT json
            FROM superset_action_logs
            WHERE slice_id = ? AND action IN ('ChartDataRestApi.data', 'ChartDataRestApi.json_dumps')
            ORDER BY superset_log_id DESC
            LIMIT 1
            """,
            [context.chart_id],
        ).fetchone()
        if not row or not row[0]:
            return {"error": "no chart log payload found"}
        try:
            payload = json.loads(row[0])
        except json.JSONDecodeError:
            return {"error": "log payload is not valid JSON"}
        return {"chart_id": context.chart_id, "sql": payload.get("sql") or payload.get("query")}

    @function_tool
    def get_chart_metadata() -> dict[str, Any]:
        """
        Return chart metadata for the active chart via Superset API.

        Input:
        - Uses the active context set by the API handler (chart id + settings).

        Output:
        - {"chart_id": int, "metadata": dict | null}
        - {"error": "..."} if dashboard id is missing or chart not found.
        """
        context = _get_active_context()
        if context is None:
            return {"error": "active context not set"}
        if not context.chart_id:
            return {"error": "active chart id not set"}
        payload = _fetch_latest_chart_log_payload(context.conn, context.chart_id)
        dashboard_id = payload.get("dashboard_id") if isinstance(payload, dict) else None
        if not dashboard_id:
            return {"error": "dashboard id not found in log payload"}
        charts = fetch_dashboard_charts(context.settings, int(dashboard_id))
        for chart in charts:
            if str(chart.get("slice_id")) == str(context.chart_id):
                return {"chart_id": context.chart_id, "metadata": chart}
        return {"chart_id": context.chart_id, "metadata": None}


class AgentRunner:
    def __init__(self) -> None:
        self._router_agent = self._build_agents()

    @property
    def available(self) -> bool:
        return self._router_agent is not None

    def _build_agents(self) -> Any:
        if Agent is None or ModelSettings is None:
            return None

        model_settings = ModelSettings(
            reasoning={"effort": "low"},
            verbosity="low",
            max_turns=30,
            response_format={"type": "json_object", "schema": Answer.model_json_schema()},
        )
        normal_agent = Agent(
            name="Normal Agent",
            model=os.getenv("AGENT_MODEL", "gpt-5-nano"),
            tools=[],
            model_settings=model_settings,
            instructions=(
                "You are a friendly assistant for general conversation. "
                "Return a JSON object with an 'answer' field containing the response."
            ),
        )

        dashboard_agent = Agent(
            name="Dashboard Agent",
            model=os.getenv("AGENT_MODEL", "gpt-5-nano"),
            tools=[
                get_active_chart_log,
                get_active_chart_data,
                get_chart_sql,
                get_chart_metadata,
            ]
            if function_tool
            else [],
            model_settings=model_settings,
            instructions=(
                "You answer questions about the current dashboard and charts. "
                "Use the provided chart context when available. "
                "Return a JSON object with an 'answer' field containing the response."
            ),
        )

        return Agent(
            name="Customer-facing agent",
            model=os.getenv("AGENT_MODEL", "gpt-5-nano"),
            tools=[
                normal_agent.as_tool(
                    tool_name="normal_expert",
                    tool_description="Handles general chat and greetings.",
                ),
                dashboard_agent.as_tool(
                    tool_name="dashboard_expert",
                    tool_description="Handles dashboard questions and requests.",
                ),
            ],
            model_settings=model_settings,
            instructions=(
                "Handle all direct user communication. "
                "Route to the dashboard expert when the user asks about charts, "
                "metrics, filters, or dashboard content; otherwise use the normal expert. "
                "Return a JSON object with an 'answer' field containing the response."
            ),
        )

    async def respond(
        self,
        message: str,
        history: list[dict[str, str]],
        context: str | None = None,
        context_obj: Any | None = None,
        debug: bool = False,
    ) -> tuple[str, list[dict[str, Any]]]:
        if not self._router_agent:
            return (
                "Agent not available. "
                "Install the OpenAI agents package and set OPENAI_API_KEY."
            ), [{"type": "error", "message": "agent_not_available"}] if debug else []

        prompt = self._format_prompt(message, history, context=context)
        debug_items: list[dict[str, Any]] = []
        try:
            global _ACTIVE_CONTEXT
            _ACTIVE_CONTEXT = context_obj
            result = await self._run_agent(self._router_agent, prompt, context_obj)
        except Exception as exc:
            if debug:
                debug_items.append({"type": "error", "message": str(exc)})
            return "Agent call failed. Check API credentials and logs.", debug_items
        finally:
            _ACTIVE_CONTEXT = None
        answer = self._extract_answer(result)
        if debug:
            debug_items.extend(self._extract_debug_items(result))
        return answer, debug_items

    def _format_prompt(
        self,
        message: str,
        history: list[dict[str, str]],
        context: str | None = None,
    ) -> str:
        lines = []
        if context:
            lines.append("context: " + context)
        for item in history[-20:]:
            role = (item.get("role") or "").strip()
            content = (item.get("content") or "").strip()
            if not role or not content:
                continue
            lines.append(f"{role}: {content}")
        if not (
            history
            and (history[-1].get("role") or "").strip() == "user"
            and (history[-1].get("content") or "").strip() == message.strip()
        ):
            lines.append(f"user: {message}")
        return "\n".join(lines)

    async def _run_agent(self, agent: Any, prompt: str, context_obj: Any | None) -> Any:
        if Runner is None:
            raise RuntimeError(_IMPORT_ERROR or "agents Runner unavailable")

        run_sync = getattr(Runner, "run_sync", None)
        if callable(run_sync):
            return await asyncio.to_thread(run_sync, agent, prompt, context=context_obj)

        run_async = getattr(Runner, "run", None)
        if callable(run_async):
            result = run_async(agent, prompt, context=context_obj)
            if asyncio.iscoroutine(result):
                return await result
            return result

        raise RuntimeError("agents Runner has no run method")

    def _extract_answer(self, result: Any) -> str:
        if isinstance(result, str):
            return self._parse_json_answer(result)

        for attr in ("final_output", "output_text", "output"):
            value = getattr(result, attr, None)
            if isinstance(value, str) and value.strip():
                return self._parse_json_answer(value)
        if isinstance(result, dict):
            return self._parse_json_answer(json.dumps(result))
        return str(result)

    def _parse_json_answer(self, text: str) -> str:
        try:
            payload = json.loads(text)
        except json.JSONDecodeError:
            return text
        if isinstance(payload, dict):
            answer = payload.get("answer") or payload.get("message") or payload.get("text")
            if isinstance(answer, str) and answer.strip():
                return answer
        return text

    def _extract_debug_items(self, result: Any) -> list[dict[str, Any]]:
        items = []
        new_items = getattr(result, "new_items", None)
        if not isinstance(new_items, list):
            return items
        for item in new_items:
            item_type = getattr(item, "type", None) or "unknown_item"
            if item_type == "reasoning_item":
                items.append({"type": item_type, "detail": "OpenAI hides the content"})
                continue
            raw = getattr(item, "raw_item", None)
            if item_type == "tool_call_item":
                name = getattr(raw, "name", None)
                if name is None and isinstance(raw, dict):
                    name = raw.get("name")
                items.append({"type": item_type, "name": name or "unknown_tool"})
                continue
            if item_type == "tool_call_output_item":
                output = None
                if isinstance(raw, dict):
                    output = raw.get("output")
                items.append({"type": item_type, "output": output})
                continue
            items.append({"type": item_type})
        return items

In [9]:
model_settings = ModelSettings(
    reasoning={"effort": "low"},
    verbosity="low",
    max_turns=30,
    response_format={"type": "json_object", "schema": Answer.model_json_schema()},
)
dashboard_agent = Agent(
    name="Dashboard Agent",
    model="gpt-5-nano",
    tools=[
        get_active_chart_log,
        get_active_chart_data,
        get_chart_sql,
        get_chart_metadata,
    ]
    if function_tool
    else [],
    model_settings=model_settings,
    instructions=(
        "You answer questions about the current dashboard and charts. "
        "Use the provided chart context when available. "
        "Return a JSON object with an 'answer' field containing the response."
    ),
)

In [10]:
from agents import Runner

query = "What is the total revenue and number of stores for each marketing group's products that were sold in the second quarter of 2024."
inputs=[
    {
        "role": "user", 
        "content": query
    }
]
result = await Runner.run(dashboard_agent, inputs)

In [11]:
for item in result.new_items:
    if item.type == 'reasoning_item':
        print(f"[{item.type}]: OpenAI hides the content")
    if item.type == 'tool_call_item':
        print(f"[{item.type}]: {item.raw_item.name}")
    if item.type == 'tool_call_output_item':
        print(f"[{item.type}]: {item.raw_item['output']}")

[reasoning_item]: OpenAI hides the content
[tool_call_item]: get_chart_sql
[tool_call_item]: get_chart_metadata
[tool_call_output_item]: {'error': 'active context not set'}
[tool_call_output_item]: {'error': 'active context not set'}
[reasoning_item]: OpenAI hides the content


In [6]:
import requests

# trigger_url = "http://localhost:8000/poller/trigger"
# requests.post(trigger_url, timeout=10).raise_for_status()

url = "http://localhost:8000/superset/dashboards/12/charts"
params = {} #{"dashboard_id": 12, "limit": 50}

resp = requests.get(url, params=params, timeout=10)
resp.raise_for_status()
data = resp.json()

In [7]:
data
# 2026-01-12T03:57:06.449956

{'dashboard_id': 12,
 'charts': [{'chart_id': 314,
   'slice_id': 314,
   'name': 'Sales and Reach by Marketing Group',
   'viz_type': None,
   'datasource_id': None,
   'datasource_type': None},
  {'chart_id': 315,
   'slice_id': 315,
   'name': 'Revenue by Quarter',
   'viz_type': None,
   'datasource_id': None,
   'datasource_type': None},
  {'chart_id': 316,
   'slice_id': 316,
   'name': 'Brand Drives',
   'viz_type': None,
   'datasource_id': None,
   'datasource_type': None}]}